# Convert csv to rubustness Report

In [2]:
import pandas as pd
import sys
import logging
from pathlib import Path

# Falls du den StdoutFilter noch brauchst (filtert alles über INFO heraus)
class StdoutFilter(logging.Filter):
    def filter(self, record):
        return record.levelno <= logging.INFO

def _setup_logger(log_to_console=True, log_to_file=True, log_file_path=None):
    """Erstellt und konfiguriert einen spezifischen Logger."""
    logger = logging.getLogger("robustness_logger")
    logger.setLevel(logging.INFO)
    
    # Bestehende Handler entfernen, um doppelte Logs bei mehrfachem Aufruf zu verhindern
    logger.handlers = []
    logger.propagate = False

    # Formatter für Markdown: Nur die reine Message, sonst wird das MD-Layout zerstört
    markdown_formatter = logging.Formatter("%(message)s")

    if log_to_console:
        # Handler für stdout (INFO und darunter)
        stdout_handler = logging.StreamHandler(sys.stdout)
        stdout_handler.setLevel(logging.INFO)
        stdout_handler.addFilter(StdoutFilter())
        stdout_handler.setFormatter(markdown_formatter)
        logger.addHandler(stdout_handler)

        # Handler für stderr (WARNING und darüber)
        stderr_handler = logging.StreamHandler(sys.stderr)
        stderr_handler.setLevel(logging.WARNING)
        stderr_handler.setFormatter(markdown_formatter)
        logger.addHandler(stderr_handler)

    if log_to_file and log_file_path:
        # Ordner erstellen, falls er noch nicht existiert
        Path(log_file_path).parent.mkdir(exist_ok=True, parents=True)
        
        file_handler = logging.FileHandler(log_file_path, mode="w", encoding="utf-8")
        file_handler.setLevel(logging.INFO)
        file_handler.setFormatter(markdown_formatter)
        logger.addHandler(file_handler)

    return logger


def get_metric_value(row, dist, cat, subcat, metric):
    """Sucht intelligent nach Werten in der Zeile und gleicht Namensunterschiede (Aliase) aus."""
    # 1. Exakter Match-Versuch
    tuple_key = (dist, cat, subcat, metric)
    if tuple_key in row.index:
        return row[tuple_key]

    # 2. Flexibler Match mit Alias-Übersetzung für alte/neue Benennungen
    dist_l = dist.lower()
    cat_l = cat.lower()
    subcat_l = subcat.lower()
    metric_l = metric.lower()

    # Aliase definieren
    subcat_aliases = [subcat_l]
    if "total samples" in subcat_l or "num samples" in subcat_l:
        subcat_aliases.extend(["total samples", "num samples"])

    metric_aliases = [metric_l]
    if metric_l in ["accuracy", "score"]:
        metric_aliases.extend(["accuracy", "score"])
    elif metric_l in ["count"]:
        metric_aliases.extend(["count"])

    # Spalten durchsuchen
    for col in row.index:
        if isinstance(col, tuple) and len(col) >= 4:
            c_dist, c_cat, c_subcat, c_metric = (
                str(col[0]).lower(),
                str(col[1]).lower(),
                str(col[2]).lower(),
                str(col[3]).lower(),
            )
            if c_dist == dist_l and c_cat == cat_l:
                if any(a in c_subcat for a in subcat_aliases) and any(
                    a in c_metric for a in metric_aliases
                ):
                    return row[col]
        else:
            # Fallback für flache Strings
            col_str = str(col).lower()
            if dist_l in col_str and cat_l in col_str:
                if any(a in col_str for a in subcat_aliases) and any(
                    a in col_str for a in metric_aliases
                ):
                    return row[col]

    return "N/A"


def find_metadata_value(row, substring):
    """Sucht nach zusätzlichen Metadatenfeldern wie Gen-Anzahlen."""
    for col in row.index:
        if substring.lower() in str(col).lower():
            return row[col]
    return None


def format_classification_report(row, dist_name):
    """Extrahiert exklusiv die Zelltypen des Classification Reports und formatiert sie sauber."""
    cell_types = set()

    for col in row.index:
        col_str = str(col)
        # STRIKTE BEDINGUNG: Muss zur Distribution gehören UND Teil des Classification Reports sein
        if (
            dist_name.lower() in col_str.lower()
            and "classification report" in col_str.lower()
        ):
            if isinstance(col, tuple) and len(col) >= 3:
                subcat = col[2]
            else:
                parts = col_str.split("_")
                subcat = parts[2] if len(parts) > 2 else ""

            # Kontrollzeilen ausschließen
            if subcat and not any(
                ex in subcat.lower()
                for ex in [
                    "overall",
                    "macro avg",
                    "weighted avg",
                    "accuracy",
                    "total_accuracy",
                ]
            ):
                cell_types.add(subcat)

    cell_types = sorted(list(cell_types))

    output = []
    # Spaltenbreite auf 15 erhöht für sauberes Alignment bei "mean +- std"
    output.append(
        f"{'':>24} {'precision':>15} {'recall':>15} {'f1-score':>15} {'support':>15}"
    )
    output.append("")

    for cell in cell_types:
        p = get_metric_value(row, dist_name, "Classification Report", cell, "precision")
        r = get_metric_value(row, dist_name, "Classification Report", cell, "recall")
        f1 = get_metric_value(
            row, dist_name, "Classification Report", cell, "f1-score"
        )
        sup = get_metric_value(
            row, dist_name, "Classification Report", cell, "support"
        )
        output.append(
            f"{cell:>24} {str(p):>15} {str(r):>15} {str(f1):>15} {str(sup):>15}"
        )

    output.append("")

    # Gesamte Accuracy ermitteln
    acc = get_metric_value(row, dist_name, "Classification Report", "Overall", "Accuracy")
    if acc == "N/A":
        acc = get_metric_value(row, dist_name, "Baseline", "Overall", "Accuracy")

    total_support = get_metric_value(
        row, dist_name, "Classification Report", "weighted avg", "support"
    )
    output.append(
        f"{'accuracy':>24} {'':>15} {'':>15} {str(acc):>15} {str(total_support):>15}"
    )

    # Averages
    for avg in ["macro avg", "weighted avg"]:
        p = get_metric_value(row, dist_name, "Classification Report", avg, "precision")
        r = get_metric_value(row, dist_name, "Classification Report", avg, "recall")
        f1 = get_metric_value(
            row, dist_name, "Classification Report", avg, "f1-score"
        )
        sup = get_metric_value(row, dist_name, "Classification Report", avg, "support")
        output.append(
            f"{avg:>24} {str(p):>15} {str(r):>15} {str(f1):>15} {str(sup):>15}"
        )

    return "\n".join(output)


def print_robustness_report(csv_path, target_row="mean +- std", log_to_console=True, log_to_file=True):
    """Liest die CSV ein und loggt das perfekt rekonstruierte Protokoll an Konsole und/oder Datei."""
    
    # Pfad für die Markdown-Datei im selben Ordner wie die CSV bestimmen
    csv_path_obj = Path(csv_path)
    log_file_path = csv_path_obj.parent / "robustness_report.md"
    
    # Logger initialisieren
    logger = _setup_logger(
        log_to_console=log_to_console, 
        log_to_file=log_to_file, 
        log_file_path=log_file_path
    )

    try:
        df = pd.read_csv(csv_path_obj, header=[0, 1, 2, 3], index_col=0)
    except Exception:
        df = pd.read_csv(csv_path_obj, index_col=0)

    if target_row not in df.index:
        raise ValueError(
            f"Zeile '{target_row}' nicht gefunden. Verfügbar: {list(df.index)}"
        )

    row = df.loc[target_row]

    logger.info("--- Robustness Evaluation ---")

    # ==================== IN DISTRIBUTION ====================
    logger.info("--- In distribution testset ---")
    base_acc_id = get_metric_value(
        row, "In-Distribution", "Baseline", "Overall", "Accuracy"
    )
    logger.info(f"Baseline accuracy score {base_acc_id}\n")

    logger.info(format_classification_report(row, "In-Distribution"))
    logger.info("")

    rand_drop_id = get_metric_value(
        row, "In-Distribution", "Dropout", "Random", "Accuracy"
    )
    logger.info(f"Random dropout accuracy score {rand_drop_id}")

    total_samples_id = get_metric_value(
        row, "In-Distribution", "Consistency", "Total Samples", "Count"
    )
    logger.info(f"Total samples: {total_samples_id}")

    incons_id = get_metric_value(
        row, "In-Distribution", "Consistency", "Inconsistent Predictions", "Count"
    )
    logger.info(f"Number of inconsistent predictions: {incons_id}")

    for th in ["0.1%", "0.5%", "1.0%", "2.0%"]:
        fi_val = get_metric_value(
            row, "In-Distribution", "Dropout", f"FI_{th}", "Accuracy"
        )
        logger.info(
            f"Feature importance dropout ({th} features dropped) accuracy score {fi_val}"
        )

    # ==================== OUT OF DISTRIBUTION ====================
    logger.info("--- Out of data distribution ---")

    genes_exp = find_metadata_value(row, "expected") or "10000"
    genes_match = find_metadata_value(row, "matched") or "8693"
    train_max = find_metadata_value(row, "Training data Max") or "8.634057"
    test_max = find_metadata_value(row, "Test data Max") or "8.726716041564941"

    logger.info(f"Genes expected in training set: {genes_exp}")
    logger.info(f"Genes actually matched in test set: {genes_match}")
    logger.info(f"Training data Max-Value: {train_max}")
    logger.info(f"Test data Max-Value: {test_max}")

    base_acc_ood = get_metric_value(
        row, "Out-of-Distribution", "Baseline", "Overall", "Accuracy"
    )
    logger.info(f"Baseline accuracy score {base_acc_ood}\n")

    logger.info(format_classification_report(row, "Out-of-Distribution"))
    logger.info("")

    rand_drop_ood = get_metric_value(
        row, "Out-of-Distribution", "Dropout", "Random", "Accuracy"
    )
    logger.info(f"Random dropout accuracy score {rand_drop_ood}")

    total_samples_ood = get_metric_value(
        row, "Out-of-Distribution", "Consistency", "Total Samples", "Count"
    )
    logger.info(f"Total samples: {total_samples_ood}")

    incons_ood = get_metric_value(
        row, "Out-of-Distribution", "Consistency", "Inconsistent Predictions", "Count"
    )
    logger.info(f"Number of inconsistent predictions: {incons_ood}")

    for th in ["0.1%", "0.5%", "1.0%", "2.0%"]:
        fi_val = get_metric_value(
            row, "Out-of-Distribution", "Dropout", f"FI_{th}", "Accuracy"
        )
        logger.info(
            f"Feature importance dropout ({th} features dropped) accuracy score {fi_val}"
        )
        
    if log_to_file:
        print(f"[Info] Report erfolgreich gespeichert unter: {log_file_path}")


# --- AUFRUF ---
# print_robustness_report("deine_datei.csv", target_row="mean +- std")
print_robustness_report("results/custom_ensemble_linsvc_sampled/combined_result.csv", target_row="mean +- std")

--- Robustness Evaluation ---
--- In distribution testset ---
Baseline accuracy score 0.915 +- 0.0051

                               precision          recall        f1-score         support

                  B cell      1.0 +- 0.0   0.9917 +- 0.0   0.9958 +- 0.0    120.0 +- 0.0
          CD14+ monocyte 0.9914 +- 0.0022 0.9997 +- 0.0005 0.9955 +- 0.0012   2575.0 +- 0.0
             CD4+ T cell 0.8709 +- 0.0033 0.9956 +- 0.0007 0.929 +- 0.0022   3910.0 +- 0.0
        Cytotoxic T cell 0.9513 +- 0.0045 0.6138 +- 0.0234  0.746 +- 0.019   1824.0 +- 0.0
          Dendritic cell      1.0 +- 0.0      0.6 +- 0.0     0.75 +- 0.0      5.0 +- 0.0
           Megakaryocyte      1.0 +- 0.0      1.0 +- 0.0      1.0 +- 0.0      7.0 +- 0.0
     Natural killer cell 0.8471 +- 0.0261 0.9198 +- 0.0033 0.8818 +- 0.0149    791.0 +- 0.0
             Plasma cell      1.0 +- 0.0      1.0 +- 0.0      1.0 +- 0.0     49.0 +- 0.0

                accuracy                                 0.915 +- 0.0051   9281.0 +-